# f696 开发期局部训练标签母本身份审计

限定读取两份已封存Qlib来源；只比较决策前原始三资产提案及日期，不读取未来标签、模型输出或收益。

In [1]:
import os,sys,hashlib,json
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path.cwd()
while not (root/'src').exists() and root!=root.parent:root=root.parent
sys.path.insert(0,str(root))
card=root/'research/stage1_f696_training_label_parent_identity_audit_20261025'
plan=json.loads((card/'plan.json').read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'result.json').exists()
for card_key,summary_key in [('f696_lineage_card','f696_lineage_summary_sha256'),('f696_card','f696_summary_sha256'),('physical_reference_card','physical_reference_summary_sha256')]:
 assert sha(root/s[card_key]/'summary.json')==s[summary_key]
assert sha(root/s['provider_uri']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
fpath=root/s['f696_card']/'qlib_recorder'/s['f696_experiment_id']/s['f696_recorder_id']/'artifacts/pit_constituent_auxiliary_training_allocation_retrained_20260928/predictions.csv'
rpath=root/s['physical_reference_card']/'qlib_recorder'/s['physical_reference_experiment_id']/s['physical_reference_recorder_id']/'artifacts/inputs/source_frame'
assert sha(fpath)==s['f696_predictions_sha256'] and sha(rpath)==s['physical_source_frame_artifact_sha256']
lineage=json.loads((root/s['f696_lineage_card']/'summary.json').read_text())
ref_summary=json.loads((root/s['physical_reference_card']/'summary.json').read_text())
assert lineage['source_legacy_full_Recorder_NAS_scoped_restore_verified'] and lineage['NAS_readback']['verified']
assert ref_summary['nas_readback']['verified'] and json.loads((root/s['physical_reference_card']/ref_summary['nas_scoped_restore']).read_text())['all_sha256_and_sizes_match']
f=pd.read_csv(fpath,usecols=['signal_date','segment','soxl_weight','soxx_weight','cash_weight'])
f=f.loc[f.segment.eq('development_walk_forward')].drop(columns='segment').copy()
f.signal_date=pd.to_datetime(f.signal_date).dt.normalize()
qlib.init(provider_uri=str((root/s['provider_uri']).resolve()),region='us',kernels=1)
with R.uri_context(str((root/s['physical_reference_card']/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['physical_reference_experiment_id'],recorder_id=s['physical_reference_recorder_id'])
 assert rec.status=='FINISHED'
 r=rec.load_object('inputs/source_frame')[['signal_date','soxl_weight','soxx_weight','cash_weight']].copy()
r.signal_date=pd.to_datetime(r.signal_date).dt.normalize()
assert len(f)==1756 and len(r)==1061 and f.signal_date.is_unique and r.signal_date.is_unique
m=r.merge(f,on='signal_date',how='outer',validate='one_to_one',suffixes=('_physical','_f696'),indicator=True).sort_values('signal_date')
common=m.loc[m._merge.eq('both')].copy();assert len(common)==1056
names=['soxl_weight','soxx_weight','cash_weight']
diffs={name:np.abs(common[name+'_physical'].to_numpy(float)-common[name+'_f696'].to_numpy(float)) for name in names}
rowmax=np.max(np.column_stack([diffs[x] for x in names]),axis=1)
result={'decision':'old_physical_training_label_parent_mismatch_exact_f696','plan_sha256':sha(card/'plan.json'),'f696_recorder_id':s['f696_recorder_id'],'physical_reference_recorder_id':rec.id,'f696_development_rows':len(f),'physical_reference_rows':len(r),'matched_signal_dates':len(common),'f696_only_dates':int(m._merge.eq('right_only').sum()),'physical_reference_only_dates':[str(x.date()) for x in m.loc[m._merge.eq('left_only'),'signal_date']],'rows_any_difference_gt_1e_8':int((rowmax>1e-8).sum()),'rows_all_three_exact_equal':int(np.logical_and.reduce([diffs[x]==0 for x in names]).sum()),'weight_differences':{name:{'exact_equal_rows':int((diffs[name]==0).sum()),'max_abs':float(diffs[name].max()),'mean_abs':float(diffs[name].mean())} for name in names},'source_NAS_scoped_readbacks_verified':True,'future_label_reads':0,'model_fits':0,'accounts':0,'new_Recorders':0,'interpretation':'Original-architecture 2018-2022 physical local labels are not matched exact-f696 OOF labels; preserve all fixed original-scaffold results in their own scope, do not relabel or automatically rerun.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_ids':[s['f696_recorder_id'],rec.id],'source_NAS_scoped_readbacks_verified':True,'matched_signal_dates':len(common),'rows_any_difference_gt_1e_8':result['rows_any_difference_gt_1e_8'],'future_label_reads':0,'model_fits':0,'accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'matched':len(common),'material_mismatch':result['rows_any_difference_gt_1e_8'],'max_abs':{k:v['max_abs'] for k,v in result['weight_differences'].items()},'physical_only':result['physical_reference_only_dates']},ensure_ascii=False))

{"matched": 1056, "material_mismatch": 1056, "max_abs": {"soxl_weight": 0.24122940246281066, "soxx_weight": 0.3663453921234032, "cash_weight": 0.5910661547054783}, "physical_only": ["2022-12-22", "2022-12-23", "2022-12-27", "2022-12-28", "2022-12-29"]}


# 局部标签母本审计：旧物理训练账本不是 f696 开发期提案

只读已封存的 f696 `development_walk_forward` 预测 CSV 与原架构2018–2022物理训练账本 `inputs/source_frame`，逐件核对SHA、来源Recorder状态及NAS限定回读；通过原生Qlib Recorder加载旧训练源。只比较信号日和SOXL/SOXX/现金三项**决策前原始权重**，0未来局部价值标签读取、0拟合、0账户。

f696开发期有1756行，旧物理参考有1061行；共同信号日 **1056**。这些共同日期的三资产权重没有任何一行逐项完全相同，且每一行至少一项差异超过1e−8。最大绝对差：SOXL **0.241229**、SOXX **0.366345**、现金 **0.591066**；平均绝对差分别为0.046739/0.059355/0.094260。旧参考独有2022-12-22、23、27、28、29五个信号日；f696另有700日。精确数字及两个Recorder ID见 `result.json`。

决定 `old_physical_training_label_parent_mismatch_exact_f696`。旧物理账本的局部标签和在其上失败的固定信息实验在**原架构范围内仍有效**，但不能充当精确 f696 联合接收器的匹配训练标签或必需信息门。尤其不能把旧局部MSE失败直接当作“f696的因子没有价值”，也不能因母本纠正自动重跑已关闭的相同字段/符号版本。

下一项有界工作应先用f696的1756行逐年向前提案与b472同日决策前状态，冻结并建立一份0bp额外滑点、IBKR Tiered费用、整股现金与自身风险递推的**精确训练参考**；核对原始三资产提案1756/1756、风险投影、动作/恢复时钟与原生Qlib账户。之后才能在同状态、次日共同目标复位下生成受限与非受限两种合法局部标签，冻结联合信息门。训练参考仍是开发证据，不是SOXX策略通过。本审计没有新Recorder、NAS来源或收益结论。
